In [5]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path(r"C:/Plegma_Programming")
DATA_PATH = BASE_DIR / "PLEGMA_final_hourly_dataset.csv"
OUT_PATH = BASE_DIR / "stores" / "history_store_home03_2026.csv"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

HOME_ID = "home03"
TIME_COL = "timestamp"
HOME_COL = "home_id"
TARGET_COL = "consumption_Wh"

# =========================================================
# LOAD
# =========================================================
usecols = [HOME_COL, TIME_COL, TARGET_COL]
df = pd.read_csv(
    DATA_PATH,
    usecols=usecols,
    parse_dates=[TIME_COL],
    low_memory=False
)

df = df.dropna(subset=[HOME_COL, TIME_COL, TARGET_COL]).copy()
df[HOME_COL] = df[HOME_COL].astype(str)
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df.dropna(subset=[TARGET_COL]).copy()

src = df[df[HOME_COL] == HOME_ID].copy().sort_values(TIME_COL)

if src.empty:
    raise ValueError(f"No data found for {HOME_ID}")

# =========================================================
# HELPER: build one mapped block
# =========================================================
def build_block(src_df, src_start, src_end, target_start, target_end):
    src_start = pd.Timestamp(src_start)
    src_end = pd.Timestamp(src_end)
    target_start = pd.Timestamp(target_start)
    target_end = pd.Timestamp(target_end)

    # source period
    block = src_df[
        (src_df[TIME_COL] >= src_start) &
        (src_df[TIME_COL] <= src_end)
    ].copy()

    # full hourly source index
    full_src_idx = pd.date_range(start=src_start, end=src_end, freq="h")

    block = block.set_index(TIME_COL).reindex(full_src_idx)
    block.index.name = TIME_COL
    block[TARGET_COL] = pd.to_numeric(block[TARGET_COL], errors="coerce")

    # fill missing for smooth demo history
    block[TARGET_COL] = block[TARGET_COL].ffill().bfill()

    # target index
    full_target_idx = pd.date_range(start=target_start, end=target_end, freq="h")

    if len(full_src_idx) != len(full_target_idx):
        raise ValueError(
            f"Length mismatch: source ({len(full_src_idx)}) vs target ({len(full_target_idx)})"
        )

    out = pd.DataFrame({
        HOME_COL: HOME_ID,
        TIME_COL: full_target_idx,
        TARGET_COL: block[TARGET_COL].astype(float).values
    })

    return out

# =========================================================
# BLOCK 1
# 2023-01-01 00:00 -> 2023-09-30 23:00
# mapped to
# 2026-01-01 00:00 -> 2026-09-30 23:00
# =========================================================
block1 = build_block(
    src_df=src,
    src_start="2023-01-01 00:00:00",
    src_end="2023-09-30 23:00:00",
    target_start="2026-01-01 00:00:00",
    target_end="2026-09-30 23:00:00",
)

# =========================================================
# BLOCK 2
# 2022-10-01 00:00 -> 2022-12-31 23:00
# mapped to
# 2026-10-01 00:00 -> 2026-12-31 23:00
# =========================================================
block2 = build_block(
    src_df=src,
    src_start="2022-10-01 00:00:00",
    src_end="2022-12-31 23:00:00",
    target_start="2026-10-01 00:00:00",
    target_end="2026-12-31 23:00:00",
)

# =========================================================
# COMBINE
# =========================================================
out = pd.concat([block1, block2], ignore_index=True)
out = out.sort_values(TIME_COL).reset_index(drop=True)

# safety check
expected_idx = pd.date_range(
    start="2026-01-01 00:00:00",
    end="2026-12-31 23:00:00",
    freq="h"
)

if len(out) != len(expected_idx):
    raise ValueError(f"Unexpected row count: {len(out)} vs expected {len(expected_idx)}")

if not out[TIME_COL].equals(pd.Series(expected_idx, name=TIME_COL)):
    raise ValueError("Timestamps are not a complete continuous hourly range for 2026")

out.to_csv(OUT_PATH, index=False, encoding="utf-8")

print("Saved:", OUT_PATH)
print("Rows:", len(out))
print("From:", out[TIME_COL].min())
print("To  :", out[TIME_COL].max())
print("\nPreview:")
print(out.head())
print(out.tail())

Saved: C:\Plegma_Programming\stores\history_store_home03_2026.csv
Rows: 8760
From: 2026-01-01 00:00:00
To  : 2026-12-31 23:00:00

Preview:
  home_id           timestamp  consumption_Wh
0  home03 2026-01-01 00:00:00       57.198025
1  home03 2026-01-01 01:00:00       62.218161
2  home03 2026-01-01 02:00:00       59.169658
3  home03 2026-01-01 03:00:00      183.024125
4  home03 2026-01-01 04:00:00      143.157346
     home_id           timestamp  consumption_Wh
8755  home03 2026-12-31 19:00:00      612.029242
8756  home03 2026-12-31 20:00:00      140.429711
8757  home03 2026-12-31 21:00:00       60.124622
8758  home03 2026-12-31 22:00:00       61.938019
8759  home03 2026-12-31 23:00:00       59.262433
